# Tournament formats: champion quality per match budget

**Question:** Which format produces a champion closest in FIFA rating points to the highest-rated team, for the matches it uses?

For each simulated tournament, **rating gap = highest input rating − champion's input rating**. We compare the average gap (lower is better) and average match count. This is a champion-selection experiment, not a test of the accuracy of a complete ranking.

The 32 teams and their strengths stay fixed. All matches use the draw-free model from notebook 01: $p(A\text{ wins}) = 1/(1+10^{-(R_A-R_B)/600})$. No home advantage, fatigue or rating updates. Conditional on ratings, match outcomes are independent. FIFA's expected-result curve is being used as a modelling assumption, not a calibrated W/L forecast.

Run all cells from the repository root or `notebooks/`. For Colab, first make the repository's `data/` directory available under the working directory. Requires **numpy, pandas and networkx**; no GPU required. If missing, install these packages in your notebook environment before running.

## Formats and rules

| Format | Seeding and advancement | Matches |
|---|---|---:|
| Knockout, random | Random draw into a fixed 32-team bracket | 31 |
| Knockout, seeded | Full FIFA seeding: 1 vs 32, 2 vs 31, etc.; 1 and 2 can meet only in the final | 31 |
| Traditional World Cup style | Four FIFA pots (1–8, 9–16, 17–24, 25–32); one random team per pot per group. Eight groups of four, single round robin; top two advance | 63 |
| Premier League style | 32-team double round robin; most wins is champion | 992 |
| League → best-of-seven playoffs | Same league stage; top **16** qualify, seeded by league standings. Four rounds; first to four wins each series | 1,052–1,097 |
| Swiss style | Adjustable rounds; all 32 play once per round; favour opponents on similar win totals, with no rematches | 16 × rounds |

**Explicit simplifications:** all standings ties are resolved by random lots among tied teams, not FIFA rating. Groups omit geographic restrictions. World Cup knockout starts A1–B2, C1–D2, E1–F2, G1–H2, B1–A2, D1–C2, F1–E2, H1–G2, keeping group partners in opposite halves. No third-place game. Playoff seeds stay fixed, with no reseeding. League matches represent two encounters, not a home advantage.

**Our Swiss variant:** start everyone on zero wins; each round choose a full pairing that minimises the sum of absolute win-count differences, excluding previous opponents. Random weights break ties between equally good pairings. Everyone plays; no byes. This is an explicit educational Swiss-style variant, not an implementation of a federation's official rules. If no rematch-free full pairing exists, it raises an error rather than silently changing rules. Final tied records use random lots; adding opponent-strength tie-breakers would be a separate experiment.


In [1]:
# EDIT THESE SETTINGS, then Run All.
N_SIMULATIONS = 1000      # Tournaments per format setting in this batch.
RANDOM_SEED = 42          # Repeatable results for unchanged settings/environment.
SWISS_ROUNDS = [5, 7, 9]  # Try [5], [4, 6, 8], etc. Each round costs 16 matches.
SCALE = 600.0
PLAYOFF_TEAMS = 16        # Keep at 16 for the selected league → playoffs design.


In [2]:
import json
import math
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import networkx as nx
from IPython.display import display

filename = "fifa-men-top32-2026-11-25.json"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data" / filename).exists()), None)
if ROOT is None:
    raise FileNotFoundError("Place the repository data/ folder in the working directory.")
snapshot = json.loads((ROOT / "data" / filename).read_text())
teams = pd.DataFrame(snapshot["teams"]).sort_values("rank").reset_index(drop=True)
names = teams.team.to_numpy()
ratings = teams.rating_points.to_numpy(dtype=float)
N = len(teams)
assert N == 32 and teams.team.is_unique
assert np.isfinite(ratings).all() and np.all(np.diff(ratings) <= 0)
assert isinstance(N_SIMULATIONS, int) and N_SIMULATIONS >= 2
assert SCALE > 0 and PLAYOFF_TEAMS == 16
assert SWISS_ROUNDS and len(set(SWISS_ROUNDS)) == len(SWISS_ROUNDS)
assert all(isinstance(r, int) and 1 <= r <= N - 1 for r in SWISS_ROUNDS)

# Calculate each of the 496 matchups once; reverse by complement.
P = np.full((N, N), np.nan)  # Self-matches are not defined.
for a, b in combinations(range(N), 2):
    P[a, b] = 1 / (1 + 10 ** (-(ratings[a] - ratings[b]) / SCALE))
    P[b, a] = 1 - P[a, b]
print("Snapshot:", snapshot["page_updated_date"])
print("Source:", snapshot["source_url"])
print("Source note:", snapshot["provenance_note"])
print(f"{N} teams; {math.comb(N, 2)} unique match probabilities; scale {SCALE:g}.")


Snapshot: 2026-11-25
Source: https://inside.fifa.com/fifa-rankings/world-ranking/men
Source note: Transcribed from current rendered FIFA table, Points column, ranks 1–32. The 18 November FIFA news article places Morocco fifth; this current table places Portugal fifth, Brazil sixth and Morocco seventh. Preserves the current table as observed, not the historical 18 November release. Cause of difference is unverified.
32 teams; 496 unique match probabilities; scale 600.


## Match outcomes and seeding

A match samples a Bernoulli outcome using the precomputed probability. A best-of-seven series stops as soon as one side wins four games, so its cost varies.

The seeded bracket is constructed recursively. Its placement—not just the first-round opponent list—keeps the highest seeds apart until later rounds.


In [3]:
def play_match(a, b, rng):
    assert a != b
    return a if rng.random() < P[a, b] else b


def seed_order(size):
    """Bracket positions occupied by seed numbers (1-based)."""
    if size < 2 or size & (size - 1):
        raise ValueError("Bracket size must be a power of two >= 2")
    order = [1, 2]
    while len(order) < size:
        total = 2 * len(order) + 1
        order = [x for seed in order for x in (seed, total - seed)]
    return order


def seeded_bracket(ordered_teams):
    return [ordered_teams[s - 1] for s in seed_order(len(ordered_teams))]


def play_series(a, b, rng, best_of=1):
    assert best_of > 0 and best_of % 2 == 1
    target = best_of // 2 + 1
    wins_a = wins_b = 0
    while max(wins_a, wins_b) < target:
        if play_match(a, b, rng) == a:
            wins_a += 1
        else:
            wins_b += 1
    return (a if wins_a == target else b), wins_a + wins_b


def knockout(bracket, rng, best_of=1):
    remaining = list(bracket)
    matches = 0
    while len(remaining) > 1:
        advanced = []
        for a, b in zip(remaining[::2], remaining[1::2]):
            winner, count = play_series(a, b, rng, best_of)
            advanced.append(winner)
            matches += count
        remaining = advanced
    return remaining[0], matches


print("Seeded first round:")
bracket = seeded_bracket(list(range(N)))
display(pd.DataFrame([{"seed_a": a + 1, "team_a": names[a],
                       "seed_b": b + 1, "team_b": names[b]}
                      for a, b in zip(bracket[::2], bracket[1::2])]))


,seed_a,team_a,seed_b,team_b
0,1,Spain,32,Algeria
1,16,USA,17,Japan
2,8,Belgium,25,Canada
3,9,Netherlands,24,Ecuador
4,4,England,29,Nigeria
5,13,Mexico,20,Uruguay
6,5,Portugal,28,Côte d'Ivoire
7,12,Switzerland,21,Denmark
8,2,Argentina,31,Paraguay
9,15,Croatia,18,Senegal


Seeded first round:


## Group and league standings

Every match awards one win to its winner; no draws. Random tie-breaks do not give the tournament access to our benchmark strength ratings. A single round robin plays each pair once; a double round robin plays each pair twice.


In [4]:
def standings(ids, wins, rng):
    """Sort by wins descending; random lots within ties."""
    ids = np.asarray(ids, dtype=int)
    return ids[np.lexsort((rng.random(len(ids)), -wins[ids]))].tolist()


def round_robin(ids, rng, repeats=1):
    ids = list(ids)
    fixtures = np.array(list(combinations(ids, 2)) * repeats, dtype=int)
    a, b = fixtures[:, 0], fixtures[:, 1]
    winners = np.where(rng.random(len(fixtures)) < P[a, b], a, b)
    wins = np.bincount(winners, minlength=N)
    return standings(ids, wins, rng), len(fixtures), wins


def world_cup(rng):
    pots = [rng.permutation(np.arange(i, i + 8)) for i in range(0, N, 8)]
    groups = np.array(pots).T
    qualifiers = []
    matches = 0
    for group in groups:
        order, count, _ = round_robin(group, rng)
        qualifiers.append(order[:2])
        matches += count
    q = qualifiers
    # A1-B2, C1-D2, E1-F2, G1-H2 | B1-A2, D1-C2, F1-E2, H1-G2
    bracket = []
    for g in (0, 2, 4, 6):
        bracket.extend([q[g][0], q[g + 1][1]])
    for g in (0, 2, 4, 6):
        bracket.extend([q[g + 1][0], q[g][1]])
    champion, count = knockout(bracket, rng)
    return champion, matches + count


## Swiss: rounds are the experimental setting

A perfect matching pairs all teams once. We find the cheapest such pairing, where the main cost is the difference in current wins. Each pairing's random tie-breaking contribution totals less than 1, so it cannot override a one-win improvement in the main cost. Previous opponents are excluded.

There are 16 matches per round. More rounds collect more evidence, but do not guarantee a perfectly ordered ranking. Changing pairing or standings tie-breaks would also change the format.


In [5]:
def swiss(rounds, rng):
    wins = np.zeros(N, dtype=int)
    played = set()
    for round_number in range(1, rounds + 1):
        graph = nx.Graph()
        graph.add_nodes_from(range(N))
        for a, b in combinations(range(N), 2):
            if (a, b) not in played:
                cost = abs(int(wins[a]) - int(wins[b])) + rng.random() / (N + 1)
                graph.add_edge(a, b, weight=cost)
        pairing = nx.min_weight_matching(graph, weight="weight")
        if len(pairing) != N // 2:
            raise ValueError(f"No full rematch-free Swiss pairing at round {round_number}. "
                             "Try fewer rounds; this implementation does not backtrack.")
        for a, b in sorted(tuple(sorted(pair)) for pair in pairing):
            assert (a, b) not in played
            played.add((a, b))
            wins[play_match(a, b, rng)] += 1
    return standings(range(N), wins, rng)[0], len(played)


## Check the rules before comparing outcomes

These checks target bracket placement, match budgets and the Swiss no-rematch rule. They use a separate random generator so they do not change the experiment's results.


In [6]:
check_rng = np.random.default_rng(123)
order = seed_order(N)
assert sorted(order) == list(range(1, N + 1))
assert all(a + b == N + 1 for a, b in zip(order[::2], order[1::2]))
assert order.index(1) // (N // 2) != order.index(2) // (N // 2)
assert len({order.index(s) // (N // 4) for s in range(1, 5)}) == 4
assert np.allclose((P + P.T)[~np.eye(N, dtype=bool)], 1)
assert knockout(seeded_bracket(list(range(N))), check_rng)[1] == 31
assert world_cup(check_rng)[1] == 63
league_order, league_count, league_wins = round_robin(range(N), check_rng, repeats=2)
assert league_count == 992 and league_wins.sum() == 992
assert sorted(league_order) == list(range(N))
_, playoff_count = knockout(seeded_bracket(league_order[:16]), check_rng, best_of=7)
assert 60 <= playoff_count <= 105
assert swiss(5, check_rng)[1] == 80
print("Bracket placement, complementary probabilities and match budgets checked.")


Bracket placement, complementary probabilities and match budgets checked.


## Run the comparison

Pause before running: which format do you predict will have the smallest average gap? Which will offer the best improvement per extra match?

The league-only and league-plus-playoffs results share the **same league season within each simulation**. This lets us ask what appending playoffs does to that season's champion quality. Other formats have separate random streams. Repeated tournaments are independent within each format; format results need not be independent of each other.

This batch uses 1,000 repetitions per format setting. Increasing repetitions reduces Monte Carlo estimation noise; it does not improve the tournament itself or validate the match model. Swiss matching costs more computation than simple brackets.


In [7]:
records = []
# Separate deterministic streams keep existing format results stable when adding Swiss settings.
format_ids = {"Knockout (random)": 1, "Knockout (seeded)": 2,
              "World Cup (seeded groups)": 3, "League": 4, "Playoffs": 5}
streams = {label: np.random.default_rng(np.random.SeedSequence([RANDOM_SEED, i]))
           for label, i in format_ids.items()}
swiss_streams = {r: np.random.default_rng(np.random.SeedSequence([RANDOM_SEED, 100, r]))
                for r in SWISS_ROUNDS}


def record(simulation, label, champion, matches):
    records.append({"simulation": simulation, "format": label,
                    "champion": names[champion], "champion_rank": int(teams.iloc[champion]["rank"]),
                    "champion_rating": ratings[champion],
                    "rating_gap": ratings.max() - ratings[champion], "matches": matches})


for simulation in range(1, N_SIMULATIONS + 1):
    rng = streams["Knockout (random)"]
    record(simulation, "Knockout (random)", *knockout(rng.permutation(N), rng))
    record(simulation, "Knockout (seeded)",
           *knockout(seeded_bracket(list(range(N))), streams["Knockout (seeded)"]))
    record(simulation, "World Cup (seeded groups)", *world_cup(streams["World Cup (seeded groups)"]))
    order, count, _ = round_robin(range(N), streams["League"], repeats=2)
    record(simulation, "League", order[0], count)
    champion, extra = knockout(seeded_bracket(order[:PLAYOFF_TEAMS]), streams["Playoffs"], best_of=7)
    record(simulation, "League + best-of-seven", champion, count + extra)
    for rounds in SWISS_ROUNDS:
        record(simulation, f"Swiss ({rounds} rounds)", *swiss(rounds, swiss_streams[rounds]))

results = pd.DataFrame(records)
assert len(results) == N_SIMULATIONS * (5 + len(SWISS_ROUNDS))
assert results.rating_gap.ge(0).all()
print(f"Completed {len(results):,} tournaments across {results['format'].nunique()} format settings.")


Completed 8,000 tournaments across 8 format settings.


## Read the results

**Mean rating gap** is our primary outcome. **Gap SD** describes variation between individual tournaments; **MC standard error = gap SD / √repetitions** describes uncertainty in the estimated mean. Approximate 95% Monte Carlo intervals use mean ± 1.96 × SE; they quantify simulation noise under this model, not uncertainty about true team strengths. The normal approximation can be weak for small or highly concentrated samples.

Match count is shown beside quality rather than hidden inside a combined score. A gap/matches ratio would reward spending extra matches even if quality did not improve, so we do not use it. Mean champion rank and #1 win rate are secondary descriptions.


In [8]:
summary = results.groupby("format", sort=False).agg(
    simulations=("rating_gap", "size"),
    mean_matches=("matches", "mean"), min_matches=("matches", "min"), max_matches=("matches", "max"),
    mean_rating_gap=("rating_gap", "mean"), gap_sd=("rating_gap", "std"),
    mean_champion_rank=("champion_rank", "mean"),
    top_rated_win_rate=("champion_rank", lambda x: (x == 1).mean()),
)
summary["gap_mc_se"] = summary.gap_sd / np.sqrt(summary.simulations)
summary["gap_ci_low"] = summary.mean_rating_gap - 1.96 * summary.gap_mc_se
summary["gap_ci_high"] = summary.mean_rating_gap + 1.96 * summary.gap_mc_se
summary = summary.sort_values("mean_matches")
display(summary.round(3))

# Compare appending playoffs to the exact same league, using paired differences.
wide = results.pivot(index="simulation", columns="format", values="rating_gap")
difference = wide["League + best-of-seven"] - wide["League"]
mean_difference = difference.mean()
se_difference = difference.std(ddof=1) / np.sqrt(len(difference))
print(f"Playoffs minus league: mean gap change {mean_difference:.2f} points")
print(f"Approximate paired 95% MC interval: "
      f"[{mean_difference - 1.96 * se_difference:.2f}, {mean_difference + 1.96 * se_difference:.2f}]")
print("Positive means appending playoffs produced a larger average gap in this run.")


,simulations,mean_matches,min_matches,max_matches,mean_rating_gap,gap_sd,mean_champion_rank,top_rated_win_rate,gap_mc_se,gap_ci_low,gap_ci_high
format,,,,,,,,,,,
Knockout (random),1000,31.000,31,31,131.971,126.157,6.385,0.198,3.989,124.152,139.790
Knockout (seeded),1000,31.000,31,31,106.853,111.639,5.049,0.225,3.530,99.934,113.773
World Cup (seeded groups),1000,63.000,63,63,124.011,121.027,5.882,0.195,3.827,116.510,131.513
Swiss (5 rounds),1000,80.000,80,80,135.650,130.249,6.598,0.202,4.119,127.577,143.723
Swiss (7 rounds),1000,112.000,112,112,118.069,119.206,5.659,0.214,3.770,110.681,125.458
Swiss (9 rounds),1000,144.000,144,144,103.335,109.721,4.934,0.225,3.470,96.535,110.136
League,1000,992.000,992,992,29.056,30.883,2.023,0.440,0.977,27.142,30.970
League + best-of-seven,1000,1074.738,1062,1087,43.314,51.272,2.508,0.334,1.621,40.136,46.492


Playoffs minus league: mean gap change 14.26 points
Approximate paired 95% MC interval: [10.54, 17.97]
Positive means appending playoffs produced a larger average gap in this run.


## Save the tables and the settings

Each run replaces these output files. Save a copy or change `OUTPUT_DIR` if you want to preserve multiple runs. Full precision is saved; rounding above is for display only. The input FIFA snapshot is never overwritten.


In [9]:
OUTPUT_DIR = ROOT / "outputs" / "tournament-comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_DIR / "tournament_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "format_summary.csv")
settings = {"n_simulations": N_SIMULATIONS, "random_seed": RANDOM_SEED,
            "swiss_rounds": SWISS_ROUNDS, "scale": SCALE, "playoff_teams": PLAYOFF_TEAMS,
            "snapshot_file": filename, "snapshot_metadata": {k: v for k, v in snapshot.items() if k != "teams"},
            "assumptions": "Fixed strengths, independent W/L matches, neutral venues, random standings ties.",
            "swiss_pairing": "Minimum total absolute wins difference, random tie weights, no rematches; no backtracking.",
            "paired_comparison": "League and league-plus-playoffs share each simulated league season.",
            "versions": {"numpy": np.__version__, "pandas": pd.__version__, "networkx": nx.__version__}}
(OUTPUT_DIR / "settings.json").write_text(json.dumps(settings, indent=2, ensure_ascii=False) + "\n")
assert len(pd.read_csv(OUTPUT_DIR / "tournament_results.csv")) == len(results)
print("Saved results, summary and settings to", OUTPUT_DIR.relative_to(ROOT))


Saved results, summary and settings to outputs/tournament-comparison


## Explore one change at a time

1. **Swiss rounds:** change `SWISS_ROUNDS` and rerun. How much does each extra round improve average champion quality for its 16-match cost?
2. **Seeding:** compare the two 31-match knockout formats. Their budgets match, so the difference isolates our bracket draw policy under the same match model.
3. **Playoffs:** compare the paired league results. Does adding games in this particular way improve or worsen selection?
4. **Estimation precision:** increase `N_SIMULATIONS`. Compare the standard errors, not just changes in the estimated means.

We have not chosen a target acceptable gap, so this notebook does not declare an “optimal” format or minimum sufficient budget. The league and World Cup comparisons deliberately have different budgets. Results describe these chosen rules, including tie-breaks; they do not establish a universal winner among formats.
